# Task 16: TabPFN v2 模型侧实验 — 10 折 CV OOF + 配对 AM 比较

**目标**: 在 64 维特征 (50PC+7struct+4ddg+3TMD) 上对比 XGBoost / TabPFN v2 / Ensemble，
用 **10 折 StratifiedGroupKFold pooled OOF** 评估，并做 **逐 seed 配对比较 vs AlphaMissense**。

**关键改动 (vs 初版)**:
1. **不用 10% test** — 用 10 折 CV pooled OOF，全量 ~2179 个预测上算一个 AUROC/AUPRC，消除小测试集方差。
2. **配对同折比 AM** — 每个 seed 内 AM AUROC 在相同的 fold assignment 上计算，TabPFN−AM 差值逐 seed 配对，不拿 3-seed 均值去比全局常数。

**红线**:
- StratifiedGroupKFold(n_splits=10, groups=Gene, shuffle=True, random_state=seed)
- PCA 在每 fold 的训练集上 fit → transform test fold（无泄漏）
- AM 不需要训练，每个 test fold 直接用 AM score 对该折样本算 AUROC（pooled 后即全量 AM AUROC）
- TabPFN v2 + 本地 ckpt，不走 HuggingFace

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from scipy.stats import ttest_1samp

warnings.filterwarnings("ignore")

# ===== 配置 =====
DEVICE     = "cuda"
MODEL_PATH = "/mnt/volume6/czj/labLGN/LabLZ/models/tabpfn-v2-classifier.ckpt"
SEEDS      = [42, 7, 2024]
N_COMP     = 50
N_FOLDS    = 10                                   # 10 折 CV OOF

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载基础特征矩阵 =====
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")

ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]
esm_cols   = [c for c in feat_cols if c.startswith("esm_")]
STRUCT_NAMES = ["plddt","sasa","rsa","ss_helix","ss_strand","ss_coil",
                "delta_hydrophobicity"]
DDG_NAMES  = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]
TMD_NAMES  = ["in_TMD", "dist_to_nearest_TMD", "delta_membrane_insertion"]

X_full_arr = df_feat[feat_cols].values.astype(np.float32)
esm_idx    = [feat_cols.index(c) for c in esm_cols]
struct_idx = [feat_cols.index(c) for c in STRUCT_NAMES if c in feat_cols]
X_esm   = X_full_arr[:, esm_idx]
X_struct = X_full_arr[:, struct_idx]

y      = (df_feat["reloc_v3"].values > 0).astype(int)
groups = df_feat["Gene"].values
full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()

print(f"n={len(y)}, pos={int(y.sum())}, base_rate={y.sum()/len(y):.4f}")

# ===== AlphaMissense =====
am_scores = df_main["AlphaMissense score"].values.astype(float)
am_valid  = np.isfinite(am_scores)               # 有 126 个 NaN，只在 valid 子集上算 AM AUROC
print(f"AM scores: {am_valid.sum()}/{len(am_scores)} 有效, {len(am_scores) - am_valid.sum()} NaN")

# ===== 加载 4 种 ddg =====
from collections import OrderedDict
ddg_files = OrderedDict([
    ("ddg_esm2",   ("ddg_esm2.csv",   "ddg_esm2")),
    ("ddg_struct", ("ddg_struct.csv", "ddg_struct")),
    ("ddg_rasp",   ("ddg_rasp.csv",   "ddg_rasp")),
    ("ddg_foldx",  ("ddg_foldx.csv",  "ddg_foldx")),
])
ddg_sources = OrderedDict()
for name, (fname, dcol) in ddg_files.items():
    df_d = pd.read_csv(BASE_PATH + fname)
    ddg_map = dict(zip(df_d["KEY"], df_d[dcol]))
    arr = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
    ddg_sources[name] = arr.reshape(-1, 1)
    print(f"  {name:12s}: {np.isfinite(arr).sum()}/{len(arr)} 有效")

# ===== 加载 TMD 特征 =====
df_tmd = pd.read_csv(BASE_PATH + "tmd_features.csv")
tmd_sources = OrderedDict()
for name in TMD_NAMES:
    tmd_map = dict(zip(df_tmd["KEY"], df_tmd[name]))
    arr = np.array([tmd_map.get(k, 0.0) for k in full_keys], dtype=np.float32)
    tmd_sources[name] = arr.reshape(-1, 1)
    print(f"  {name:25s}: {(arr != 0).sum()}/{len(arr)} 非零")

print(f"\n特征: {N_COMP}PC + {len(struct_idx)}struct + {len(DDG_NAMES)}ddg + {len(TMD_NAMES)}TMD = "
      f"{N_COMP + len(struct_idx) + len(DDG_NAMES) + len(TMD_NAMES)} 维")
print(f"评估: {N_FOLDS} 折 CV OOF × {len(SEEDS)} seeds")

n=2179, pos=235, base_rate=0.1078
AM scores: 2053/2179 有效, 126 NaN
  ddg_esm2    : 2179/2179 有效
  ddg_struct  : 2168/2179 有效
  ddg_rasp    : 2168/2179 有效
  ddg_foldx   : 2166/2179 有效
  in_TMD                   : 151/2179 非零
  dist_to_nearest_TMD      : 664/2179 非零
  delta_membrane_insertion : 151/2179 非零

特征: 50PC + 7struct + 4ddg + 3TMD = 64 维
评估: 10 折 CV OOF × 3 seeds


In [2]:
def run_one_seed(seed):
    """单 seed 10 折 CV: 每折训练 XGBoost + TabPFN，收集 OOF 预测。
    
    Returns:
        oof_preds: dict with keys "XGBoost", "TabPFN", "Ensemble", "AM"
                   每个 value 是 (n,) array — 全量 OOF 预测 (与输入同序)
        per_fold:  list of dict, 每折的 AUROC
    """
    print(f"\n{'='*65}")
    print(f"  Seed {seed} — {N_FOLDS} 折 StratifiedGroupKFold CV")
    print(f"{'='*65}")
    
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    
    # OOF 预测缓冲区 (按原始索引回填)
    oof_xgb  = np.full(len(y), np.nan, dtype=np.float32)
    oof_tab  = np.full(len(y), np.nan, dtype=np.float32)
    
    per_fold = []
    
    for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full_arr, y, groups)):
        y_tr, y_te = y[tr_idx], y[te_idx]
        
        # --- ESM2 → Impute → Scale → PCA (fit on train only) ---
        imp_e = SimpleImputer(strategy="median")
        scl_e = StandardScaler()
        Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
        Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
        pca = PCA(n_components=N_COMP, random_state=42)
        Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
        Xe_te_pca = pca.transform(Xe_te).astype(np.float32)
        
        # --- Struct ---
        imp_s = SimpleImputer(strategy="median")
        scl_s = StandardScaler()
        Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
        Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)
        
        parts_tr = [Xe_tr_pca, Xs_tr]
        parts_te = [Xe_te_pca, Xs_te]
        
        # --- ddg ---
        for name in DDG_NAMES:
            imp_d = SimpleImputer(strategy="median")
            parts_tr.append(imp_d.fit_transform(ddg_sources[name][tr_idx]).astype(np.float32))
            parts_te.append(imp_d.transform(ddg_sources[name][te_idx]).astype(np.float32))
        
        # --- TMD ---
        for name in TMD_NAMES:
            imp_t = SimpleImputer(strategy="median")
            parts_tr.append(imp_t.fit_transform(tmd_sources[name][tr_idx]).astype(np.float32))
            parts_te.append(imp_t.transform(tmd_sources[name][te_idx]).astype(np.float32))
        
        X_tr = np.hstack(parts_tr).astype(np.float32)
        X_te = np.hstack(parts_te).astype(np.float32)
        
        assert not np.isnan(X_tr).any() and not np.isnan(X_te).any(),                f"Fold {fold}: NaN after imputation!"
        
        # --- XGBoost ---
        scale_weight = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
        sw = compute_sample_weight("balanced", y_tr)
        xgb = XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.5,
            scale_pos_weight=scale_weight,
            objective="binary:logistic", eval_metric="aucpr",
            random_state=seed, n_jobs=-1, tree_method="hist")
        xgb.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        oof_xgb[te_idx] = xgb.predict_proba(X_te)[:, 1]
        
        # --- TabPFN v2 ---
        from tabpfn import TabPFNClassifier
        tab = TabPFNClassifier(device=DEVICE, model_path=MODEL_PATH)
        tab.fit(X_tr, y_tr)
        oof_tab[te_idx] = tab.predict_proba(X_te)[:, 1]
        
        # --- 每折 AUROC (快速诊断) ---
        auc_xgb = roc_auc_score(y_te, oof_xgb[te_idx])
        auc_tab = roc_auc_score(y_te, oof_tab[te_idx])
        per_fold.append({"fold": fold, "n_test": len(te_idx),
                         "xgb_auroc": auc_xgb, "tab_auroc": auc_tab})
        
        print(f"  Fold {fold}: n_test={len(te_idx):4d}  "
              f"XGB={auc_xgb:.4f}  TabPFN={auc_tab:.4f}  "
              f"Δ={auc_tab - auc_xgb:+.4f}")
    
    # --- Ensemble: rank 平均 ---
    oof_ens = (pd.Series(oof_xgb).rank().values + pd.Series(oof_tab).rank().values) / 2
    
    oof_preds = {
        "XGBoost":  oof_xgb,
        "TabPFN":   oof_tab,
        "Ensemble": oof_ens,
        "AM":       am_scores,                     # AM 无 OOF 概念，全量 score
    }
    
    # 全量 OOF 汇总
    aucs = {}
    for name, preds in oof_preds.items():
        if name == "AM":
            mask = am_valid & np.isfinite(preds)   # AM 只算 valid 子集
        else:
            mask = np.isfinite(preds)
        aucs[name] = {
            "auroc": roc_auc_score(y[mask], preds[mask]),
            "auprc": average_precision_score(y[mask], preds[mask]),
            "n":     mask.sum(),
        }
    
    print(f"  --- OOF 全量 ---")
    for name in ["XGBoost", "TabPFN", "Ensemble", "AM"]:
        a = aucs[name]
        print(f"  {name:10s}  AUROC={a['auroc']:.4f}  AUPRC={a['auprc']:.4f}  (n={a['n']})")
    
    return oof_preds, per_fold, aucs

In [3]:
# ===== 多 seed 运行 =====
all_seed_preds = {}   # seed → oof_preds dict
all_seed_folds = {}   # seed → per_fold list
all_seed_aucs  = {}   # seed → aucs dict

for seed in SEEDS:
    oof_preds, per_fold, aucs = run_one_seed(seed)
    all_seed_preds[seed] = oof_preds
    all_seed_folds[seed] = per_fold
    all_seed_aucs[seed]  = aucs

print(f"\n全部 {len(SEEDS)} seeds 完成.")


  Seed 42 — 10 折 StratifiedGroupKFold CV
  Fold 0: n_test= 193  XGB=0.6448  TabPFN=0.6478  Δ=+0.0031
  Fold 1: n_test= 235  XGB=0.5700  TabPFN=0.5667  Δ=-0.0033
  Fold 2: n_test= 221  XGB=0.6641  TabPFN=0.6769  Δ=+0.0128
  Fold 3: n_test= 212  XGB=0.6984  TabPFN=0.7630  Δ=+0.0645
  Fold 4: n_test= 195  XGB=0.6519  TabPFN=0.6541  Δ=+0.0022
  Fold 5: n_test= 218  XGB=0.5842  TabPFN=0.6264  Δ=+0.0422
  Fold 6: n_test= 235  XGB=0.7018  TabPFN=0.7342  Δ=+0.0324
  Fold 7: n_test= 216  XGB=0.6415  TabPFN=0.6431  Δ=+0.0017
  Fold 8: n_test= 229  XGB=0.5484  TabPFN=0.6338  Δ=+0.0854
  Fold 9: n_test= 225  XGB=0.6841  TabPFN=0.7431  Δ=+0.0590
  --- OOF 全量 ---
  XGBoost     AUROC=0.6397  AUPRC=0.1837  (n=2179)
  TabPFN      AUROC=0.6641  AUPRC=0.2319  (n=2179)
  Ensemble    AUROC=0.6663  AUPRC=0.2233  (n=2179)
  AM          AUROC=0.6362  AUPRC=0.1622  (n=2053)

  Seed 7 — 10 折 StratifiedGroupKFold CV
  Fold 0: n_test= 207  XGB=0.5421  TabPFN=0.5852  Δ=+0.0431
  Fold 1: n_test= 242  XGB=0.5547  T

In [4]:
# ===== 汇总表: per-seed AUROC/AUPRC =====
print(f"\n{'='*90}")
print(f"  Task 16: 10 折 CV OOF — XGBoost vs TabPFN vs Ensemble vs AlphaMissense")
print(f"  n={len(y)}, pos={int(y.sum())}, base_rate={y.sum()/len(y):.4f}")
print(f"  SEEDS={SEEDS},  N_FOLDS={N_FOLDS}")
print(f"{'='*90}")

records = []
for seed in SEEDS:
    aucs = all_seed_aucs[seed]
    for model in ["XGBoost", "TabPFN", "Ensemble", "AM"]:
        records.append({
            "seed": seed, "model": model,
            "AUROC": aucs[model]["auroc"],
            "AUPRC": aucs[model]["auprc"],
        })

df_rep = pd.DataFrame(records)

# mean ± std across seeds
summary = df_rep.groupby("model")[["AUROC","AUPRC"]].agg(["mean","std"]).round(4)
print(summary.to_string())
print(f"{'='*90}")

# ===== 配对比较: 逐 seed TabPFN − AM =====
print(f"\n--- 配对比较 (逐 seed TabPFN/Ensemble/XGBoost − AM) ---")
for model in ["XGBoost", "TabPFN", "Ensemble"]:
    deltas = []
    for seed in SEEDS:
        model_auc = all_seed_aucs[seed][model]["auroc"]
        am_auc    = all_seed_aucs[seed]["AM"]["auroc"]
        deltas.append(model_auc - am_auc)
    
    deltas = np.array(deltas)
    mean_d = deltas.mean()
    std_d  = deltas.std(ddof=1)
    
    # 单样本 t 检验 (双尾): H0: Δ=0
    if len(deltas) >= 3:
        t_stat, p_val = ttest_1samp(deltas, 0.0)
        sig = " ***" if p_val < 0.001 else (" **" if p_val < 0.01 else (" *" if p_val < 0.05 else ""))
        print(f"  {model:10s}  Δ = {deltas[0]:+.4f}, {deltas[1]:+.4f}, {deltas[2]:+.4f}  "
              f"→ mean={mean_d:+.4f} ± {std_d:.4f}  (t={t_stat:.2f}, p={p_val:.4f}{sig})")
    else:
        print(f"  {model:10s}  Δ = {deltas[0]:+.4f}, {deltas[1]:+.4f}, {deltas[2]:+.4f}  "
              f"→ mean={mean_d:+.4f} ± {std_d:.4f}")
    print(f"            per-seed: " + ", ".join([f"seed{s}={v:+.4f}" for s, v in zip(SEEDS, deltas)]))

# ===== 折间方差 (pooled across seeds) =====
print(f"\n--- 折间 AUROC 波动 (pooled across seeds) ---")
for model_key, label in [("xgb_auroc", "XGBoost"), ("tab_auroc", "TabPFN")]:
    all_fold_aucs = []
    for seed in SEEDS:
        for fd in all_seed_folds[seed]:
            all_fold_aucs.append(fd[model_key])
    all_fold_aucs = np.array(all_fold_aucs)
    print(f"  {label:10s}  fold-level: mean={all_fold_aucs.mean():.4f}, "
          f"std={all_fold_aucs.std():.4f}, "
          f"range=[{all_fold_aucs.min():.4f}, {all_fold_aucs.max():.4f}]")

# ===== 最佳模型 =====
best_model = summary["AUROC"]["mean"].idxmax()
best_auroc = summary.loc[best_model, ("AUROC", "mean")]
print(f"\n  最佳模型: {best_model} (mean AUROC={best_auroc:.4f})")
print(f"  AM mean AUROC: {summary.loc['AM', ('AUROC', 'mean')]:.4f}")


  Task 16: 10 折 CV OOF — XGBoost vs TabPFN vs Ensemble vs AlphaMissense
  n=2179, pos=235, base_rate=0.1078
  SEEDS=[42, 7, 2024],  N_FOLDS=10
           AUROC           AUPRC        
            mean     std    mean     std
model                                   
AM        0.6362  0.0000  0.1622  0.0000
Ensemble  0.6507  0.0149  0.2104  0.0114
TabPFN    0.6518  0.0134  0.2229  0.0078
XGBoost   0.6221  0.0157  0.1765  0.0062

--- 配对比较 (逐 seed TabPFN/Ensemble/XGBoost − AM) ---
  XGBoost     Δ = +0.0035, -0.0267, -0.0192  → mean=-0.0141 ± 0.0157  (t=-1.56, p=0.2599)
            per-seed: seed42=+0.0035, seed7=-0.0267, seed2024=-0.0192
  TabPFN      Δ = +0.0279, +0.0014, +0.0176  → mean=+0.0157 ± 0.0134  (t=2.03, p=0.1801)
            per-seed: seed42=+0.0279, seed7=+0.0014, seed2024=+0.0176
  Ensemble    Δ = +0.0302, +0.0005, +0.0129  → mean=+0.0145 ± 0.0149  (t=1.68, p=0.2344)
            per-seed: seed42=+0.0302, seed7=+0.0005, seed2024=+0.0129

--- 折间 AUROC 波动 (pooled across seeds) 

## 16a. 结论

**判据 (按优先级)**:
1. **Ensemble > XGBoost 且 Δ 显著 (p<0.05)** → 白捡增益，采用集成。
2. **TabPFN ≈ 或 < XGBoost** → 模型侧到顶，天花板在特征侧。
3. **折间 std 很大 (>0.05)** → 数据划分敏感，任何"提升"需谨慎解读。

**与初版 8:1:1 的差异**:
- 初版 10% test (~218 样本) 的 AUROC 方差大；现在 10 折 OOF 拼成全量 ~2179 预测，每个 seed 的估计稳定得多。
- 配对比较消掉了"拿 3-seed 均值比全局常数"的伪精度。

## 16b. 下一步

- 若 Ensemble 有效 → task17: 5 折 CV 协议下验证集成稳健性。
- 若模型侧到顶 → 投入 DeepLoc 2.0 的 WT/MT delta 特征。
- 若折间方差大 → 增加 n_splits 或换 RepeatedStratifiedGroupKFold。